## RAG Evelution Using Faiss 

In [14]:
# Load Data 
import warnings 
warnings.filterwarnings('ignore')
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('Static GK 2025.pdf')
pages = loader.load()

In [15]:
!uv pip install langchain-nvidia-ai-endpoints
!uv pip install openrouter

Using Python 3.12.13 environment at: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv
Checked 1 package in 42ms
Using Python 3.12.13 environment at: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv
Checked 1 package in 4ms


In [16]:
# Prepare and split data 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import hashlib

spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=180)
text_spliter = spliter.split_documents(pages)
chunks = [i.page_content for i in text_spliter]
metadata = [i.metadata for i in text_spliter]
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]
print(f'print first 5 ids : {ids[:2]}')

print first 5 ids : ['df52eef7bfa55759b4642211e13e3020', '622d6c3b19974d6f39f9950848df1607']


In [17]:
# Create Embedding 
from sentence_transformers import SentenceTransformer 
embedding = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
encode_chunks = embedding.encode(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
dimension = encode_chunks.shape[1]
print(f'the dimension is : {dimension}')

the dimension is : 384


In [19]:
# Using Indexing search 
import faiss  
indexing = faiss.IndexFlatL2(dimension)
indexing.add(encode_chunks)
print(f'sucessfully : {indexing}')

sucessfully : <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x131b5a270> >


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:1.5b", 
    temperature=0
)


In [21]:
# Retrive Data 
def Retrive_Data(query:str):
    prompt = f""" write the query based on symentic search : {query} """
    query_re = llm.invoke(prompt).content
    query_embedding = embedding.encode([query_re])
    dis , docs = indexing.search(x=query_embedding,k=3)
    threshold = 1.5
    print(f'the distance is : {dis[0]}')
    dense_docs = []
    for i , d in zip(dis[0],docs[0]):
        if threshold > i :
            dense_docs.append(chunks[d])
    return dense_docs

def Generate_answer(question:str , contexts_list : list ):
    if not contexts_list:
        return "NOT RELATED CONTENT"
    content_string = "\n\n".join(contexts_list)
    
    question = "what is the largest country in the world? "
    prompt = f""" You are a AI assistent so provided user's asking questions answers based on 
    local given document .
    content : {content_string}
    question : {question}
    """
    # Generate 
    result = llm.invoke(prompt)
    return result.content

In [22]:
!uv pip install langchain_huggingface

Using Python 3.12.13 environment at: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv
Checked 1 package in 35ms


In [23]:
!uv pip install "langchain-community<0.4"

Using Python 3.12.13 environment at: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv
Checked 1 package in 14ms


In [27]:
from datasets import Dataset 
from ragas import evaluate 
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_huggingface import HuggingFaceEmbeddings 

qus = "what is the largest country in the world?"
Ground_truth = "the largest country in the world by area is **Russia**"

retrive_list = Retrive_Data(query=qus)
final_answer = Generate_answer(question=qus, contexts_list=retrive_list)

data = {
    "user_input": [qus],              
    "response": [final_answer],        
    "retrieved_contexts": [retrive_list], 
    "reference": [Ground_truth]            
}
dataset = Dataset.from_dict(data)

ragas_embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") 

# Use lowercase metric instances here
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=llm,
    embeddings=ragas_embedding
)
print(result)

the distance is : [0.90092677 1.1292048  1.1869138 ]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'answer_relevancy': 1.0000, 'context_precision': 1.0000, 'context_recall': 1.0000}
